# 🐍 Aula 18 — Gestão de Fila em GPU: versão enxuta (complemento da Aula 15)

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem
> 📌 **Complemento da [Aula 15](../../aula15/README.md).** Demo enxuta no Colab (Linux)
> + **laboratório em Python** com lock por diretório, sem Bash.

O laboratório tem **1 GPU** e **4 alunos** que precisam treinar modelos ao mesmo tempo. Sem controle, os jobs concorrem pelo mesmo recurso, corrompem resultados e causam `CUDA Out of Memory (OOM)`. O time precisa de um sistema de fila automatizado com suporte a prioridade: jobs de alta prioridade executam primeiro, outros aguardam na fila — tudo com scripts de fila (Bash no Colab; alternativa Python no laboratório).

In [ ]:
# 1. Verificar GPU e ambiente disponível no Colab
!nvidia-smi || echo "NVIDIA GPU não detectada. Executando em modo CPU/Simulado."

In [ ]:
%%writefile train_job.py
# 2. Criar script train_job.py para os testes de concorrência
import argparse, time, random, os
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

parser = argparse.ArgumentParser()
parser.add_argument("--nome", default="Job", help="Nome do job")
parser.add_argument("--epocas", type=int, default=3)
parser.add_argument("--gpu", type=int, default=0)
args = parser.parse_args()

pid = os.getpid()
device = f"cuda:{args.gpu}" if HAS_TORCH and torch.cuda.is_available() else "cpu"
print(f"[{args.nome}] PID={pid} | Dispositivo={device} | Épocas={args.epocas}")

for epoca in range(1, args.epocas + 1):
    if HAS_TORCH and torch.cuda.is_available():
        a = torch.randn(2048, 2048, device=device)
        b = torch.randn(2048, 2048, device=device)
        c = torch.mm(a, b)
        torch.cuda.synchronize()
    loss = 1.0 / (epoca * random.uniform(0.8, 1.2))
    print(f"[{args.nome}] Época {epoca}/{args.epocas} | loss={loss:.4f}")
    time.sleep(random.uniform(0.5, 1.2))
print(f"[{args.nome}] Concluído com sucesso!")


In [ ]:
%%writefile gpu_queue.sh
# 3. Criar o script de fila com flock em Bash (gpu_queue.sh)
#!/usr/bin/env bash
set -euo pipefail

PRIORIDADE="${1:-2}"
NOME_JOB="${2:-job_$$}"
SCRIPT="${3:-train_job.py}"
shift 3 2>/dev/null || true

QUEUE_DIR="gpu_queue_spool"
LOCK_FILE="gpu_queue.lock"
LOG_FILE="gpu_queue.log"
mkdir -p "$QUEUE_DIR"

TICKET="${PRIORIDADE}_$(date +%Y%m%d_%H%M%S_%N)_${NOME_JOB}"
echo "$SCRIPT $*" > "$QUEUE_DIR/$TICKET"
echo "[$(date '+%H:%M:%S')] Job '$NOME_JOB' (prio=${PRIORIDADE}) enfileirado: $TICKET" | tee -a "$LOG_FILE"

while true; do
    PROXIMO=$(ls "$QUEUE_DIR" 2>/dev/null | sort | head -1 || true)
    if [ "$PROXIMO" = "$TICKET" ]; then break; fi
    sleep 2
done

(
  flock -x 200
  echo "[$(date '+%H:%M:%S')] Executando '$NOME_JOB'" | tee -a "$LOG_FILE"
  python3 "$SCRIPT" "$@"
  rm -f "$QUEUE_DIR/$TICKET"
) 200>"$LOCK_FILE"


In [ ]:
# 4. Executar o teste lançando 4 jobs simultâneos
!chmod +x gpu_queue.sh
import subprocess, time

print("Lançando 4 jobs concorrentes...")
jobs = [
    ["./gpu_queue.sh", "1", "Job-Alta-A",  "train_job.py", "--nome", "Job-Alta-A",  "--epocas", "3"],
    ["./gpu_queue.sh", "3", "Job-Baixa-B", "train_job.py", "--nome", "Job-Baixa-B", "--epocas", "2"],
    ["./gpu_queue.sh", "2", "Job-Media-C", "train_job.py", "--nome", "Job-Media-C", "--epocas", "4"],
    ["./gpu_queue.sh", "1", "Job-Alta-D",  "train_job.py", "--nome", "Job-Alta-D",  "--epocas", "2"],
]
procs = [subprocess.Popen(cmd) for cmd in jobs]
for p in procs:
    p.wait()
print("Todos os jobs concluídos!")

In [ ]:
# 5. Exibir os logs de execução da fila
!cat gpu_queue.log || true

---  
## 📝 7. Exercícios Práticos (5)  
Resolva os 5 exercícios a seguir completando as células de código correspondentes:

### Exercício 1: Fila com Prioridade Personalizada  
Adicione um 5° job com prioridade `1` após 2 segundos de atraso e verifique se ele assume a execução antes dos jobs de prioridade 2 e 3.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# Exemplo: subprocess.Popen(['./gpu_queue.sh', '1', 'Job-Urgente-E', ...])

### Exercício 2: Simulação de Aging (Prevenção de Starvation)  
Escreva um trecho Python ou Bash que verifique se algum ticket está na fila há mais de 10 segundos e reduza o seu número de prioridade (aumentando a prioridade do job).

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# Verifique os arquivos em gpu_queue_spool e renomeie o ticket antigo.

### Exercício 3: Ajuste de Prioridade de Processo (`nice`)  
Execute um job Python utilizando o comando `nice -n 15` e verifique como a prioridade de CPU é registrada pelo SO.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# Exemplo: !nice -n 15 python3 train_job.py --nome "Job-Nice"

### Exercício 4: Verificação de Lock Não-Bloqueante (`flock -n`)  
Crie um script que tente adquirir o lock com `flock -n` e, se a GPU estiver ocupada, exiba uma mensagem amigável em vez de ficar esperando.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# Exemplo: flock -n gpu.lock -c 'python3 train_job.py' || echo 'GPU ocupada no momento.'

### Exercício 5: Monitor de Fila Ativa  
Escreva uma função em Python que liste todos os arquivos em `gpu_queue_spool`, formatando a saída com o nome do job e o nível de prioridade.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# Exemplo: os.listdir('gpu_queue_spool')...

---  
## 📌 8. Síntese e Conclusão  

- O `flock` garante exclusão mútua simples para impedir colisões de VRAM e CUDA OOM.
- O sistema de tickets por arquivo (prioridade + timestamp) estabelece uma fila FIFO estruturada por classe de prioridade.
- Ferramentas como `nice`, `ionice` e `systemd` complementam o gerenciamento garantindo isolamento de CPU, I/O e relançamento automático.